# CLM v0.4 — 双 T4、Seed17、六模型各2M Pilot

选择 GPU T4 ×2、打开Internet，沿用 GITHUB_TOKEN。Save & Run All。
CPU/Controller/目标精度/GPU因果缓存门禁 → 实测双Worker六模型吞吐 → Pilot Dry Run → 预测通过才启动恰好12M有效targets。
每张GPU一个独立进程，动态领取模型；不使用DDP/DataParallel。少于两张T4或预测超预算会停止，不减Targets。
总墙钟硬限120分钟（含环境、数据、门禁、训练、评估、发布）。跨会话恢复验证不再执行，只复用既有恢复设施。
单Seed Pilot不是正式三Seed科学确认；不会自动进入8M/32M。


In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

BRANCH = 'codex/clm-v04-pilot-2m-seed17'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-stateful-routing-001'
OUTPUT = WORKSPACE / 'epclm-stateful-computation-routing-001-bootstrap'
HARD_LIMIT_SECONDS = 7200
started = time.monotonic()
PLATFORM_RESERVE_SECONDS = 60
deadline = started + HARD_LIMIT_SECONDS - PLATFORM_RESERVE_SECONDS

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(process.pid, signal.SIGTERM)
        else: process.terminate()
        try: output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix': os.killpg(process.pid, signal.SIGKILL)
            else: process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Notebook time limit reached; partial reports and final models, if any, remains under ' + str(OUTPUT))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Output is under ' + str(OUTPUT))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'switch', '-C', BRANCH, 'FETCH_HEAD'])
    run_bounded(['git', '-C', str(REPO), 'clean', '-fd'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH,
        REPOSITORY, str(REPO)])

SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT = WORKSPACE / ('clm-v04-pilot-' + SOURCE_COMMIT[:12])

import importlib.util
missing = [name for name in ('datasets','tokenizers','huggingface_hub','pytest') if importlib.util.find_spec(name) is None]
if missing:
    run_bounded([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing, timeout_seconds=600)
run_bounded([sys.executable, str(REPO / 'research/projects/cellular-language-models/experiments/clm-workspace-controller-001/pilot.py'),
    '--output', str(OUTPUT), '--started', str(started)], cwd=REPO)
